# UMAP

In [6]:
import numpy as np
import pandas as pd
import re

import umap
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
fingergen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
import plotly.graph_objects as go
import plotly.io as pio

In [7]:
#importing data for umap
base_dir = '/Users/emily/Library/CloudStorage/OneDrive-ImperialCollegeLondon/Peptide Research - Shared/Data/git_data_vf/final_training_data'

new_1M = pd.read_csv(f'{base_dir}/pretraining/new_1M.csv')
hemo = pd.read_csv(f'{base_dir}/chemberta_hemolysis_chemprop_550.csv')
camsol = pd.read_csv(f'{base_dir}/even_chemberta_noncanonical_camsol_chemprop_10k.csv')
syn = pd.read_csv(f'{base_dir}/synthesize_chemprop.csv')

In [8]:
#canonical categorizing

CANONICAL_AAS = set("ACDEFGHIKLMNPQRSTVWY")
SPECIAL_TOKENS = {"[AMD]"}

def categorise_sequence(seq: str):
    bracketed = set(re.findall(r'\[[^\]]+\]', seq))
    noncanonical_brackets = bracketed - SPECIAL_TOKENS
    if noncanonical_brackets:
        return "Non-natural"
    stripped = re.sub(r'\[[^\]]+\]', '', seq)
    noncanonical_chars = set(stripped) - CANONICAL_AAS
    if noncanonical_chars:
        return "Non-natural"

    return "Natural"

def smiles_to_fingerprint(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return fingergen.GetFingerprintAsNumPy(mol)

In [ ]:
def add_fingerprints(df, seq_col='linear_SMILES'):
    df = df.copy()
    df['Morgan_linear'] = df[seq_col].apply(smiles_to_fingerprint)
    return df

def add_umap_cols(df, embedding, index):
    return pd.concat([df, pd.DataFrame(embedding, index=index,
                                        columns=['UMAP_0_expfirst', 'UMAP_1_expfirst'])], axis=1)

# Combine experimental sets and tag canonical status
exp = pd.concat([hemo, syn, camsol], ignore_index=True)
exp['canonical_status'] = exp['Consolidated_Sequences'].apply(categorise_sequence)
new_1M['canonical_status'] = "Natural"
all_data = pd.concat([new_1M, exp], ignore_index=True)

# Fit UMAP on experimental (non-generated) data
exp_only = add_fingerprints(all_data[all_data['class'] != 'generated_canonical'])
X_exp = np.vstack(exp_only['Morgan_linear'])

reducer = umap.UMAP(n_components=2, metric='jaccard', random_state=42, verbose=True)
reducer.fit(X_exp)
exp_umap = add_umap_cols(exp_only, reducer.embedding_, exp_only.index)

# Project generated sequences into the same UMAP space
gen_can = add_fingerprints(all_data[all_data['class'] == 'generated_canonical'])
X_gen = np.vstack(gen_can['Morgan_linear'])
gen_umap = add_umap_cols(gen_can, reducer.transform(X_gen), gen_can.index)

new_all = pd.concat([gen_umap, exp_umap], ignore_index=True)

/opt/miniconda3/envs/chemprop/lib/python3.14/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for jaccard distance metric; inverse_transform will be unavailable
  warn(
/opt/miniconda3/envs/chemprop/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


UMAP(angular_rp_forest=True, metric='jaccard', n_jobs=1, random_state=42, verbose=True)
Thu Aug  6 09:45:52 2026 Construct fuzzy simplicial set
Thu Aug  6 09:45:52 2026 Finding Nearest Neighbors
Thu Aug  6 09:45:52 2026 Building RP forest with 11 trees
Thu Aug  6 09:45:54 2026 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	 3  /  14
	 4  /  14
	 5  /  14
	Stopping threshold met -- exiting after 5 iterations
Thu Aug  6 09:46:02 2026 Finished Nearest Neighbor Search
Thu Aug  6 09:46:03 2026 Construct embedding


Epochs completed:   4%| ▍          9/200 [00:00]

	completed  0  /  200 epochs


Epochs completed:  14%| █▍         29/200 [00:01]

...

	completed  160  /  200 epochs


Epochs completed:  92%| █████████▎ 185/200 [00:05]

	completed  180  /  200 epochs


Epochs completed: 100%| ██████████ 200/200 [00:06]


Thu Aug  6 09:46:10 2026 Finished embedding


In [ ]:
#umap plot

pio.defaults.default_timeout = 360
axes_size_small = 16
CLASS_COLORS = {
    "generated_canonical": {"Natural": "#F1F7B5", "Non-natural": "#F1F7B5"},
    "hemolysis_smiles":          {"Natural": "#C0392B", "Non-natural": "#F1948A"},
    "synthesize_smiles":         {"Natural": "#1E8449", "Non-natural": "#82E0AA"},
    "noncanonical_camsol_smiles": {"Natural": "#1A5276", "Non-natural": "#7FB3D3"},
}

# marker size per class - generated_canonical gets bumped up
CLASS_MARKER_SIZE = {
    "generated_canonical": 18,
    "hemolysis_smiles": 10,
    "synthesize_smiles": 10,
    "noncanonical_camsol_smiles": 10,
}

fig = go.Figure()

PLOT_ORDER = [
    ("generated_canonical",        "Natural"),
    ("noncanonical_camsol_smiles", "Non-natural"),
    ("hemolysis_smiles",           "Non-natural"),
    ("synthesize_smiles",          "Non-natural"),
    ("noncanonical_camsol_smiles", "Natural"),
    ("hemolysis_smiles",           "Natural"),
    ("synthesize_smiles",          "Natural"),
]



for cls, canon_val in PLOT_ORDER:
    mask = (new_all["class"] == cls) & (new_all["canonical_status"] == canon_val)
    sub  = new_all[mask]
    if sub.empty:
        continue
    color = CLASS_COLORS.get(cls, {}).get(canon_val, "grey")
    size  = CLASS_MARKER_SIZE.get(cls, 10)
    fig.add_trace(go.Scatter(
        x=sub["UMAP_0_expfirst"],
        y=sub["UMAP_1_expfirst"],
        mode="markers",
        name=f"{cls} ({canon_val})",
        marker=dict(color=color, size=size, opacity=0.7, line=dict(width=0.5, color="black")),
    ))

fig.update_layout(
    xaxis_title="UMAP 0",
    yaxis_title="UMAP 1",
    width=700,
    height=900,
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        mirror=True,
        tickfont=dict(size=axes_size_small, family="Arial"), title_standoff=5,
        title_font=dict(size=18, family="Arial"),
        ticks="outside"
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        mirror=True,
        tickfont=dict(size=16, family="Arial"), title_standoff=5,
        title_font=dict(size=18, family="Arial"),
        ticks="outside"
    ),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.12,
        xanchor="center",
        x=0.5,
        font=dict(size=18, family="Arial"),
    ))
fig.update_layout(xaxis=dict(range=[5, 20]), yaxis=dict(range=[0, 14]))

#plots alot of datapoints so svg is too much
fig.write_image("umap_with_pretrained_exp_first_nogen_test.png", scale = 10)